# Apex Retail Intelligence

## Notebook 2 : Bronze Layer

### Objective

The objective of this notebook is to create the Bronze Layer using the Parquet files generated in the Landing Layer.

In this notebook, I will:

- Read the historical and incremental Parquet files from the Landing Layer.
- Store the data in Delta format.
- Add an ingestion timestamp to every record.
- Store historical and incremental data separately.
- Preserve all columns without any transformation.

The Bronze Layer stores raw business data in Delta format and serves as the source for the Silver Layer.

## Project Pipeline

```text
                Apex Retail Intelligence Pipeline

 Historical CSV          Incremental CSV
        │                      │
        └──────────┬───────────┘
                   ▼
            Raw Landing Layer
                   │
                   ▼
             Bronze Layer
            (Delta Storage)
                   │
                   ▼
              Silver Layer
      (Cleaning + MERGE + SCD)
                   │
                   ▼
               Gold Layer
      (Dimensions & Fact Tables)
                   │
                   ▼
              KPI Dashboard
```

In [0]:
# ============================================================
# Apex Retail Intelligence
# Notebook 2 : Bronze Layer
# ============================================================

# Import required Spark SQL functions

from pyspark.sql.functions import *

# Import Spark SQL data types

from pyspark.sql.types import *

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1 : Define Bronze Layer Paths

In this step, I am defining the Landing and Bronze Layer paths.

The Landing Layer contains the Parquet files created in the previous notebook.

The Bronze Layer will store the datasets in Delta format.

In [0]:
# ============================================================
# Step 1 : Define Project Paths
# ============================================================

LANDING_PATH = "/Volumes/workspace/default/apex_retail_volume/landing"

BRONZE_PATH = "/Volumes/workspace/default/apex_retail_volume/bronze"

print("Project paths defined successfully.")

Project paths defined successfully.


## Step 2 : Read Historical Parquet Files

In this step, I am reading the historical Parquet files from the Landing Layer.

These datasets will be used to create the Bronze Layer.

In [0]:
# ============================================================
# Step 2 : Read Historical Parquet Files
# ============================================================

# Read Customer Historical Dataset

customer_bronze_df = spark.read.parquet(
    f"{LANDING_PATH}/historical/customer"
)

# Read Product Historical Dataset

product_bronze_df = spark.read.parquet(
    f"{LANDING_PATH}/historical/product"
)

# Read Sales Historical Dataset

sales_bronze_df = spark.read.parquet(
    f"{LANDING_PATH}/historical/sales"
)

print("Historical Parquet files loaded successfully.")

Historical Parquet files loaded successfully.


## Step 3 : Preview Historical Datasets

In this step, I am displaying the historical datasets to verify that the Parquet files have been loaded successfully.

In [0]:
# ============================================================
# Step 3 : Preview Historical Datasets
# ============================================================

display(customer_bronze_df)

display(product_bronze_df)

display(sales_bronze_df)

customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,City D,State Y
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,null,State X
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,City A,State Y
5,60,Female,null,Yes,7,Yes,Divorced,2,Bachelor's,Employed,17760,City B,State Z
6,25,Other,Medium,Yes,4,Yes,Divorced,0,Bachelor's,Unemployed,54549,City D,State Z
7,78,Male,High,No,0,No,Single,2,Master's,Retired,76235,City D,State X
8,38,null,Low,Yes,2,No,Married,1,Master's,Employed,52863,City B,State Y
9,56,Female,Low,No,0,No,Single,4,PhD,Self-Employed,65537,City C,State Y
10,75,Male,Medium,No,3,No,Married,2,High School,Self-Employed,43331,City C,State X


product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price
1480,Product D,Brand Y,Electronics,2.5,560,48,0.4,Small,4.61,Red,Metal,2019-08-04 01:47:01,2022-05-28 14:54:02,250,49.72
1597,Product C,Brand X,Groceries,4.7,413,80,0.3,Medium,0.84,Blue,Metal,2019-10-23 19:59:17,2022-12-19 08:04:41,180,817.76
5142,Product B,null,Toys,4.6,312,14,0.08,Medium,0.23,Green,Plastic,2018-05-12 08:00:29,2023-02-01 12:15:07,131,270.3
8447,Product A,Brand Z,Toys,1.1,110,69,0.09,Large,4.37,Blue,Wood,2019-11-15 16:17:29,2023-02-05 11:46:57,16,547.84
6025,Product C,Brand X,Clothing,3.8,172,25,0.39,Small,1.68,Red,Metal,2019-08-27 02:58:19,2023-10-05 08:13:07,57,785.29
1883,Product A,Brand Y,Clothing,3.3,678,74,0.38,Large,0.26,null,Glass,2019-02-09 22:23:36,2022-10-31 01:48:25,89,751.17
7781,Product D,Brand Y,Toys,2.4,434,20,0.09,Medium,9.12,White,Plastic,2018-12-14 12:35:24,2023-11-04 17:03:25,316,340.07
8642,Product B,Brand Y,Groceries,2.4,868,60,0.26,Medium,0.58,Blue,Wood,2018-01-06 22:13:14,2022-01-24 05:50:56,360,17.7
7193,Product B,Brand Z,Groceries,null,392,45,0.28,Medium,6.52,Blue,Wood,2018-12-25 09:49:50,2022-10-04 03:00:26,77,458.85
7739,Product A,Brand X,Clothing,3.6,356,61,0.03,Large,8.38,Red,Plastic,2018-02-27 19:00:28,2022-02-20 12:19:26,10,612.28


transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend
503290,2020-10-11 10:08:52,1,1480,8,49.72,0.5,Credit Card,Location A,18,Wednesday,27,7,563.16,271,20% Off,No,Spring,Yes
347796,2021-12-08 01:07:40,2,1597,7,817.76,0.32,Credit Card,Location C,15,Friday,20,2,7554.57,631,Flash Sale,No,Summer,Yes
493688,2020-02-17 09:40:48,3,5142,8,270.3,0.35,Debit Card,Location A,9,Saturday,35,6,7564.14,879,Flash Sale,Yes,Winter,Yes
861348,2020-08-13 00:43:14,4,8447,2,547.84,0.1,null,Location A,13,Friday,42,8,8125.92,211,Buy One Get One Free,Yes,Winter,No
535835,2021-07-02 11:59:03,5,6025,4,785.29,0.17,Mobile Payment,Location C,17,Monday,37,3,114.32,862,Flash Sale,Yes,Summer,Yes
978720,2020-12-01 16:31:54,6,1883,2,751.17,0.32,Credit Card,Location B,19,Saturday,50,2,3372.17,171,Buy One Get One Free,Yes,Summer,No
24070,2020-03-14 22:58:48,7,7781,5,340.07,0.41,Debit Card,Location A,15,Thursday,12,10,1322.64,664,Buy One Get One Free,No,Spring,No
752282,2021-11-17 15:56:04,8,8642,7,17.7,0.42,Debit Card,Location D,22,Saturday,3,9,1716.65,733,Buy One Get One Free,Yes,Summer,Yes
11898,2020-05-18 08:48:18,9,7193,7,458.85,0.2,Mobile Payment,Location B,19,Sunday,6,10,1358.62,429,Flash Sale,Yes,Winter,No
321956,2020-03-10 18:03:08,10,7739,4,612.28,0.1,Credit Card,Location C,19,Saturday,33,8,6757.7,516,20% Off,Yes,Spring,Yes


## Step 4 : Add Ingestion Timestamp

In this step, I am adding an ingestion timestamp to each historical dataset.

The `ingested_at` column stores the date and time when the records entered the Bronze Layer.

This metadata helps track data ingestion and supports auditing.

In [0]:
# ============================================================
# Step 4 : Add Ingestion Timestamp
# ============================================================

from pyspark.sql.functions import current_timestamp

# Add ingestion timestamp to Customer Dataset

customer_bronze_df = customer_bronze_df.withColumn(
    "ingested_at",
    current_timestamp()
)

# Add ingestion timestamp to Product Dataset

product_bronze_df = product_bronze_df.withColumn(
    "ingested_at",
    current_timestamp()
)

# Add ingestion timestamp to Sales Dataset

sales_bronze_df = sales_bronze_df.withColumn(
    "ingested_at",
    current_timestamp()
)

print("Ingestion timestamp added successfully.")

Ingestion timestamp added successfully.


In [0]:
# ============================================================
# Create Project Folder Structure
# ============================================================

dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/raw/historical")
dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/raw/incremental")
dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/raw/audit")

dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/landing/historical")
dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/landing/incremental")

dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/bronze/historical")
dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/bronze/incremental")

dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/silver")

dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/gold")

dbutils.fs.mkdirs("/Volumes/workspace/default/apex_retail_volume/kpi")

print("Project folders created successfully.")

Project folders created successfully.


## Step 5 : Verify Bronze Schema

In this step, I am checking the schema after adding the ingestion timestamp.

The new `ingested_at` column should be available in all Bronze datasets.

In [0]:
# ============================================================
# Step 5 : Verify Bronze Schema
# ============================================================

print("Customer Dataset Schema")
customer_bronze_df.printSchema()

print("\nProduct Dataset Schema")
product_bronze_df.printSchema()

print("\nSales Dataset Schema")
sales_bronze_df.printSchema()

Customer Dataset Schema
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = false)


Product Dataset Schema
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_rating: string (nullable = true)
 |-- product_review_count: string (nullable = tr

## Step 6 : Convert Historical Data into Delta Format

In this step, I am storing the historical datasets in Delta format.

The Bronze Layer stores raw business data without applying any transformations.

The historical datasets will be written in overwrite mode because this is the initial load.

In [0]:
# ============================================================
# Step 6 : Write Historical Delta Tables
# ============================================================

# Write Customer Historical Dataset

customer_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}/historical/customer")

# Write Product Historical Dataset

product_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}/historical/product")

# Write Sales Historical Dataset

sales_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}/historical/sales")

print("Historical Delta tables created successfully.")

Historical Delta tables created successfully.


## Step 7 : Read Incremental Parquet Files

In this step, I am reading the incremental Parquet files from the Landing Layer.

These datasets contain the latest records that will be appended to the Bronze Layer.

In [0]:
# ============================================================
# Step 7 : Read Incremental Parquet Files
# ============================================================

# Read Customer Incremental Dataset

customer_incremental_bronze_df = spark.read.parquet(
    f"{LANDING_PATH}/incremental/customer"
)

# Read Product Incremental Dataset

product_incremental_bronze_df = spark.read.parquet(
    f"{LANDING_PATH}/incremental/product"
)

# Read Sales Incremental Dataset

sales_incremental_bronze_df = spark.read.parquet(
    f"{LANDING_PATH}/incremental/sales"
)

print("Incremental Parquet files loaded successfully.")

Incremental Parquet files loaded successfully.


## Step 8 : Add Ingestion Timestamp to Incremental Data

In this step, I am adding the ingestion timestamp to all incremental datasets.

The timestamp records when the incremental data entered the Bronze Layer.


In [0]:
# ============================================================
# Step 8 : Add Ingestion Timestamp
# ============================================================

from pyspark.sql.functions import current_timestamp

customer_incremental_bronze_df = customer_incremental_bronze_df.withColumn(
    "ingested_at",
    current_timestamp()
)

product_incremental_bronze_df = product_incremental_bronze_df.withColumn(
    "ingested_at",
    current_timestamp()
)

sales_incremental_bronze_df = sales_incremental_bronze_df.withColumn(
    "ingested_at",
    current_timestamp()
)

print("Incremental ingestion timestamp added successfully.")

Incremental ingestion timestamp added successfully.


## Step 9 : Append Incremental Data to Bronze Layer

In this step, I am storing the incremental datasets in Delta format.

The data is written in append mode because the Bronze Layer stores every incoming record without removing duplicates.

In [0]:
# ============================================================
# Step 9 : Append Incremental Data to Bronze Layer
# ============================================================

# Append Customer Incremental Dataset

customer_incremental_bronze_df.write \
    .format("delta") \
    .mode("append") \
    .save(f"{BRONZE_PATH}/incremental/customer")

# Append Product Incremental Dataset

product_incremental_bronze_df.write \
    .format("delta") \
    .mode("append") \
    .save(f"{BRONZE_PATH}/incremental/product")

# Append Sales Incremental Dataset

sales_incremental_bronze_df.write \
    .format("delta") \
    .mode("append") \
    .save(f"{BRONZE_PATH}/incremental/sales")

print("Incremental Delta tables created successfully.")

Incremental Delta tables created successfully.


## Step 10 : Verify Bronze Layer

In this step, I am verifying that the historical and incremental Delta tables have been created successfully.

This confirms that the Bronze Layer is ready for the Silver Layer.

In [0]:
# ============================================================
# Step 10 : Verify Bronze Layer
# ============================================================

print("Historical Bronze Files")

display(dbutils.fs.ls(f"{BRONZE_PATH}/historical"))

print("Incremental Bronze Files")

display(dbutils.fs.ls(f"{BRONZE_PATH}/incremental"))

Historical Bronze Files


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/bronze/historical/customer/,customer/,0,1786208516759
dbfs:/Volumes/workspace/default/apex_retail_volume/bronze/historical/product/,product/,0,1786208516759
dbfs:/Volumes/workspace/default/apex_retail_volume/bronze/historical/sales/,sales/,0,1786208516759


Incremental Bronze Files


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/bronze/incremental/customer/,customer/,0,1786208517618
dbfs:/Volumes/workspace/default/apex_retail_volume/bronze/incremental/product/,product/,0,1786208517618
dbfs:/Volumes/workspace/default/apex_retail_volume/bronze/incremental/sales/,sales/,0,1786208517618


## Step 11 : Display Bronze Layer Summary

In this step, I am displaying a summary of the Bronze Layer.

The summary shows the number of records available in the historical and incremental Bronze datasets.

This helps verify that the data has been stored successfully in the Bronze Layer.

In [0]:
# ============================================================
# Step 11 : Display Bronze Layer Summary
# ============================================================

bronze_summary = [

    (
        "Customer Historical",
        customer_bronze_df.count()
    ),

    (
        "Product Historical",
        product_bronze_df.count()
    ),

    (
        "Sales Historical",
        sales_bronze_df.count()
    ),

    (
        "Customer Incremental",
        customer_incremental_bronze_df.count()
    ),

    (
        "Product Incremental",
        product_incremental_bronze_df.count()
    ),

    (
        "Sales Incremental",
        sales_incremental_bronze_df.count()
    )

]

bronze_summary_df = spark.createDataFrame(
    bronze_summary,
    ["Dataset", "Records"]
)

display(bronze_summary_df)

Dataset,Records
Customer Historical,1052
Product Historical,1043
Sales Historical,1002
Customer Incremental,1053
Product Incremental,1041
Sales Incremental,1000


## Step 12 : Bronze Layer Summary

The Bronze Layer has been created successfully.

The following tasks were completed:

- Read historical Parquet files from the Landing Layer.
- Read incremental Parquet files from the Landing Layer.
- Added the `ingested_at` timestamp column.
- Stored historical datasets in Delta format.
- Stored incremental datasets in Delta format using append mode.
- Verified the Bronze Layer.
- Generated the Bronze Layer summary.

The Bronze Layer is now ready for the Silver Layer.

In [0]:
# ============================================================
# Step 12 : Bronze Layer Summary
# ============================================================

print("Bronze Layer completed successfully.")
print("Bronze Layer is ready for the Silver Layer.")

Bronze Layer completed successfully.
Bronze Layer is ready for the Silver Layer.


## Notebook Summary

✔ Delta Tables Created

✔ Bronze Validation Completed

✔ Ready for Silver Layer